# LEVIR-CD  -  Data verification, visual audit & U-Net pilot

**Purpose:** Verify the downloaded LEVIR-CD dataset, inspect paired satellite images with their masks, and optionally run a tiny U-Net smoke test. **This is not the final research training pipeline.**

- Original data stays untouched.
- `train.zip` and `val.zip` are extracted into a new `_extracted/` folder if necessary.
- **Do not train on or tune against the `test` split.** We only count its files to verify download completeness.
- All outputs are written under `levir-change-detection/results/levir_audit/`.
- Do not load the downloaded third-party `.pt` / `.pth` checkpoint files for this baseline.

Run the cells from top to bottom. Edit only the **Paths & switches** cell initially.

## 1. Paths & switches

Expected structure (you can change `DATA_ROOT` if yours differs):

```text
~/SJSU/data255/datasets/levir-cd/
  train.zip
  val.zip
  test/A/, test/B/, test/label/
```

The notebook will also look for the archives inside additional `drive-download-*` folders. If you get a *missing archive* error, change `DATA_ROOT` to the common parent of your downloaded files.

In [ ]:
from pathlib import Path

import os

# Jupyter normally runs the notebook from the notebooks/ folder.
_here = Path.cwd().resolve()
_default_project = _here.parent if _here.name == "notebooks" else _here
PROJECT_ROOT = Path(os.environ.get("LEVIR_PROJECT_ROOT", str(_default_project))).expanduser().resolve()
DATA_ROOT = Path(os.environ.get("LEVIR_DATA_ROOT", str(PROJECT_ROOT.parent / "datasets" / "levir-cd"))).expanduser().resolve()
OUTPUT_DIR = PROJECT_ROOT / "results" / "levir_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTRACT_ARCHIVES = True  # extracts train.zip & val.zip; never modifies the originals
RUN_PILOT = False       # Optional tiny smoke test, not full training
PILOT_EPOCHS = 1         # short smoke test only, not full training
SEED = 1511

print("Dataset folder:", DATA_ROOT)
print("Dataset folder exists:", DATA_ROOT.exists())
print("Outputs folder:", OUTPUT_DIR)
assert DATA_ROOT.is_dir(), "DATA_ROOT not found. Put train.zip, val.zip, test/ there or change DATA_ROOT."

## 2. Locate downloaded ZIPs and inspect their contents

In [ ]:
from collections import Counter
import zipfile

archives = sorted(p for p in DATA_ROOT.rglob("*.zip") if p.is_file() and "_extracted" not in p.parts)
print("Archives found:")
for p in archives:
    print(f"  {p.relative_to(DATA_ROOT)} ({p.stat().st_size / (1024**2):.1f} MiB)")

archive_map = {}
for split in ("train", "val"):
    choices = [p for p in archives if p.name.lower() == f"{split}.zip"]
    if len(choices) > 1:
        print(f"WARNING: multiple {split}.zip files; using {choices[0]}")
    archive_map[split] = choices[0] if choices else None

for split, zpath in archive_map.items():
    if zpath is None:
        print(f"{split}.zip: not found (possibly already extracted)")
        continue
    with zipfile.ZipFile(zpath) as z:
        bad = z.testzip() if zpath.stat().st_size < 350_000_000 else None
        print(f"{split}.zip: {len(z.infolist())} entries; sample:")
        for name in z.namelist()[:8]:
            print("  ", name)
        if bad:
            raise RuntimeError(f"Damaged archive member: {bad}")

## 3. Extract training/validation archives safely

This can require several GB of free disk space. Extraction runs **once** and skips archives already extracted. It does not delete the ZIPs. The test folder stays untouched.

In [ ]:
import shutil

EXTRACT_ROOT = DATA_ROOT / "_extracted"

def safe_extract(zpath, destination):
    destination.mkdir(parents=True, exist_ok=True)
    base = destination.resolve()
    with zipfile.ZipFile(zpath) as z:
        for member in z.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(base):
                raise ValueError(f"Unsafe archive member: {member.filename}")
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError(f"Symlink not allowed: {member.filename}")
        z.extractall(destination)

for split in ("train", "val"):
    zpath = archive_map[split]
    out = EXTRACT_ROOT / split
    marker = out / ".extraction_complete"
    if zpath is None:
        print(f"{split}: no ZIP; will search for an existing extracted split")
    elif marker.exists():
        print(f"{split}: already extracted at {out}")
    elif EXTRACT_ARCHIVES:
        if out.exists() and any(out.iterdir()):
            print(f"{split}: partial folder exists; removing incomplete extraction at {out}")
            shutil.rmtree(out)
        print(f"Extracting {zpath.name}...")
        safe_extract(zpath, out)
        marker.write_text(f"source={zpath}\nsize={zpath.stat().st_size}\n")
        print(f"Done: {out}")
    else:
        print(f"{split}: extraction disabled")

## 4. Find the actual A / B / label directories

Handles ZIP archives that contain extra folder levels, as well as a directly extracted `test/` folder.

In [ ]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def valid_split_dir(d):
    return d.is_dir() and all((d / part).is_dir() for part in ("A", "B", "label"))

def locate_split(split):
    candidates = [p for p in DATA_ROOT.rglob("*") if p.is_dir() and p.name == split and valid_split_dir(p)]
    # For unusual layouts (e.g. archive root is A/B/label but named differently)
    if not candidates:
        fallback_base = EXTRACT_ROOT / split
        candidates = [p for p in fallback_base.rglob("*") if valid_split_dir(p)] if fallback_base.exists() else []
        if valid_split_dir(fallback_base):
            candidates.append(fallback_base)
    if not candidates:
        return None
    candidates.sort(key=lambda p: (0 if "_extracted" in p.parts and split in ("train","val") else 1, len(p.parts)))
    return candidates[0]

SPLIT_DIRS = {s: locate_split(s) for s in ("train", "val", "test")}
for s, d in SPLIT_DIRS.items():
    print(f"{s:5s}: {d if d else 'NOT FOUND'}")

assert SPLIT_DIRS["train"] is not None, "Training folder missing: check ZIP contents and extraction."
assert SPLIT_DIRS["val"] is not None, "Validation folder missing: check ZIP contents and extraction."

## 5. Pair images and masks; export a manifest

Checks that filenames match between `A`, `B`, and `label`. Test data is checked for existence and pair counts only; we don't use test images for tuning.

In [ ]:
import pandas as pd

def files_by_stem(folder):
    paths = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    stems = [p.stem for p in paths]
    if len(stems) != len(set(stems)):
        raise ValueError(f"Duplicate file stems in {folder}")
    return {p.stem: p for p in paths}

PAIRS = {}
records = []
for split, sd in SPLIT_DIRS.items():
    if sd is None:
        continue
    a, b, y = (files_by_stem(sd / k) for k in ("A", "B", "label"))
    names = sorted(set(a) & set(b) & set(y))
    unmatched = (set(a) | set(b) | set(y)) - set(names)
    print(f"{split}: A={len(a)}, B={len(b)}, label={len(y)}, complete triplets={len(names)}, unmatched stems={len(unmatched)}")
    if unmatched:
        print("  First unmatched stems:", sorted(unmatched)[:10])
    assert not unmatched, f"Mismatched file names in {split}; inspect before continuing."
    PAIRS[split] = [(a[name], b[name], y[name]) for name in names]
    for name in names:
        records.append({"split":split, "stem":name, "image_A":str(a[name]), "image_B":str(b[name]), "mask":str(y[name])})

manifest = pd.DataFrame(records)
manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)
print("Saved:", OUTPUT_DIR / "split_manifest.csv")
print(manifest.groupby("split").size())

## 6. Check image dimensions, labels, and class balance (train and val ONLY)

Most LEVIR-CD labels use `0` for unchanged and `255` for changed. This cell confirms what you actually downloaded. For time, the first pass samples up to 50 images per split; set `CHECK_ALL = True` for a complete audit.

In [ ]:
import numpy as np
from PIL import Image
import random
random.seed(SEED)
CHECK_ALL = True

quality_rows = []
for split in ("train", "val"):
    examples = PAIRS[split] if CHECK_ALL else random.sample(PAIRS[split], min(50, len(PAIRS[split])))
    for a_path, b_path, y_path in examples:
        with Image.open(a_path) as ia, Image.open(b_path) as ib, Image.open(y_path) as im:
            ashape, bshape, mshape = ia.size, ib.size, im.size
            mask = np.asarray(im.convert("L"))
        unique_values = np.unique(mask)
        quality_rows.append({
            "split":split, "stem":a_path.stem,
            "width":ashape[0], "height":ashape[1],
            "sizes_match":ashape == bshape == mshape,
            "mask_values":",".join(map(str, unique_values.tolist()[:12])),
            "changed_fraction":float((mask >= 128).mean()),
        })
quality = pd.DataFrame(quality_rows)
quality.to_csv(OUTPUT_DIR / "image_mask_quality_sample.csv", index=False)
print(quality.groupby("split")["changed_fraction"].agg(["count", "mean", "min", "max"]))
print("Sizes all match:", quality["sizes_match"].all())
print("Observed mask-value examples:", quality["mask_values"].value_counts().head(8).to_dict())
assert quality["sizes_match"].all(), "Some before/after/mask dimensions differ."

## 7. Display matching before / after / change-mask images

**Only training examples are displayed.** This allows you to see whether the masks align visually with real building changes.

In [ ]:
import matplotlib.pyplot as plt

train_pairs = PAIRS["train"]
selected = random.sample(train_pairs, min(35, len(train_pairs)))
scored = []
for row in selected:
    with Image.open(row[2]) as im:
        ratio = (np.asarray(im.convert("L")) >= 128).mean()
    scored.append((ratio, row))
scored.sort(key=lambda x: x[0], reverse=True)
examples = [pair for _, pair in scored[:min(3,len(scored))]]

fig, axes = plt.subplots(len(examples), 3, figsize=(12, 4*len(examples)), squeeze=False)
for i, (a_path, b_path, m_path) in enumerate(examples):
    with Image.open(a_path) as im: a = np.asarray(im.convert("RGB"))
    with Image.open(b_path) as im: b = np.asarray(im.convert("RGB"))
    with Image.open(m_path) as im: mask = np.asarray(im.convert("L")) >= 128
    for j, arr in enumerate((a,b,mask)):
        axes[i,j].imshow(arr, cmap="gray" if j == 2 else None, vmin=0 if j == 2 else None, vmax=1 if j == 2 else None)
        axes[i,j].axis("off")
    for j, title in enumerate(("Before (A)", "After (B)", "Change mask (label)")):
        axes[i,j].set_title(f"{title} | {a_path.stem}" if j == 0 else title)
plt.tight_layout()
figure_path = OUTPUT_DIR / "training_examples.png"
plt.savefig(figure_path, dpi=130, bbox_inches="tight")
plt.show()
print("Saved:", figure_path)

## 8. Sanity-check the splits (filenames only)

Same filename in two splits is a **warning**, not automatically leakage. Confirm image or location identity if filenames overlap. LEVIR-CD provides geospatial metadata, which can help with stronger location-level checks later.

In [ ]:
from itertools import combinations
for a, b in combinations(PAIRS, 2):
    fa = {p[0].stem for p in PAIRS[a]}
    fb = {p[0].stem for p in PAIRS[b]}
    common = sorted(fa & fb)
    print(f"{a} vs {b}: {len(common)} shared stems", common[:8])
print("Dataset preparation/audit complete. Test images have not been visualized or used for model selection.")

---
## 9. Optional: one-epoch 6-channel U-Net smoke test

Change `RUN_PILOT = True` in the first cell and rerun this cell (or Run All). Requires PyTorch. This is only to verify that paired images can flow through a model end-to-end.

- **Not a meaningful final score.** Uses a tiny training/validation subset and just one epoch.
- Does **not** use the test split or downloaded pretrained checkpoints.
- Next research experiments should use full training data, multiple seeds, documented training budgets, and a frozen test protocol.

In [ ]:
if not RUN_PILOT:
    print("Pilot skipped. Set RUN_PILOT=True near the top when ready.")
else:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    print("PyTorch:", torch.__version__)
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    class PairedChangeDataset(Dataset):
        def __init__(self, pairs, crop=256, repeat=1, random_crop=False):
            self.pairs = list(pairs)
            self.crop = crop
            self.repeat = repeat
            self.random_crop = random_crop
        def __len__(self):
            return len(self.pairs) * self.repeat
        def __getitem__(self, idx):
            a_path, b_path, y_path = self.pairs[idx % len(self.pairs)]
            with Image.open(a_path) as im: a = np.asarray(im.convert("RGB"), dtype=np.uint8)
            with Image.open(b_path) as im: b = np.asarray(im.convert("RGB"), dtype=np.uint8)
            with Image.open(y_path) as im: mask = np.asarray(im.convert("L"), dtype=np.uint8)
            h, w = mask.shape
            c = self.crop
            assert h >= c and w >= c, f"Image too small to crop: {y_path}"
            if self.random_crop:
                top = random.randint(0, h - c)
                left = random.randint(0, w - c)
            else:
                top, left = (h - c)//2, (w - c)//2
            a = a[top:top+c, left:left+c]
            b = b[top:top+c, left:left+c]
            mask = mask[top:top+c, left:left+c]
            x = torch.from_numpy(np.concatenate([a,b],axis=2).copy()).permute(2,0,1).float() / 255
            y = torch.from_numpy((mask >= 128).astype(np.float32).copy())[None]
            return x, y

    class ConvBlock(nn.Module):
        def __init__(self, c_in, c_out):
            super().__init__()
            self.net = nn.Sequential(nn.Conv2d(c_in,c_out,3,padding=1),nn.ReLU(inplace=True),
                                     nn.Conv2d(c_out,c_out,3,padding=1),nn.ReLU(inplace=True))
        def forward(self,x): return self.net(x)

    class TinyUNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.e1 = ConvBlock(6,12)
            self.e2 = ConvBlock(12,24)
            self.e3 = ConvBlock(24,48)
            self.mid = ConvBlock(48,96)
            self.d3 = ConvBlock(96+48,48)
            self.d2 = ConvBlock(48+24,24)
            self.d1 = ConvBlock(24+12,12)
            self.out = nn.Conv2d(12,1,1)
        def forward(self, x):
            x1=self.e1(x); x2=self.e2(F.max_pool2d(x1,2)); x3=self.e3(F.max_pool2d(x2,2))
            h=self.mid(F.max_pool2d(x3,2))
            h=self.d3(torch.cat([F.interpolate(h,size=x3.shape[-2:],mode="nearest"),x3],dim=1))
            h=self.d2(torch.cat([F.interpolate(h,size=x2.shape[-2:],mode="nearest"),x2],dim=1))
            h=self.d1(torch.cat([F.interpolate(h,size=x1.shape[-2:],mode="nearest"),x1],dim=1))
            return self.out(h)

    train_subset = random.Random(SEED).sample(PAIRS["train"], min(24, len(PAIRS["train"])))
    val_subset = random.Random(SEED+1).sample(PAIRS["val"], min(8, len(PAIRS["val"])))
    train_loader = DataLoader(PairedChangeDataset(train_subset, repeat=2, random_crop=True), batch_size=2, shuffle=True, num_workers=0)
    val_loader = DataLoader(PairedChangeDataset(val_subset, random_crop=False), batch_size=2, shuffle=False, num_workers=0)
    model = TinyUNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCEWithLogitsLoss()
    metrics=[]
    for epoch in range(PILOT_EPOCHS):
        model.train(); losses=[]
        for x, y in train_loader:
            x=x.to(device); y=y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss=criterion(model(x),y)
            loss.backward(); optimizer.step()
            losses.append(loss.item())
        model.eval(); tp=fp=fn=0
        with torch.no_grad():
            for x,y in val_loader:
                pred=(torch.sigmoid(model(x.to(device))) >= 0.5).cpu().bool()
                true=y.bool()
                tp += (pred & true).sum().item()
                fp += (pred & ~true).sum().item()
                fn += (~pred & true).sum().item()
        f1 = (2*tp)/(2*tp+fp+fn+1e-9)
        iou = tp/(tp+fp+fn+1e-9)
        metrics.append({"epoch":epoch+1,"train_loss":float(np.mean(losses)),"val_F1":f1,"val_IoU":iou})
        print(f"Pilot epoch {epoch+1}/{PILOT_EPOCHS}: loss={np.mean(losses):.4f}, val F1={f1:.4f}, val IoU={iou:.4f}")
    pd.DataFrame(metrics).to_csv(OUTPUT_DIR / "pilot_metrics_not_final.csv", index=False)
    torch.save(model.state_dict(), OUTPUT_DIR / "pilot_unet_state_dict.pt")
    print("Smoke test completed. Results are for pipeline debugging only.")

## What to send your team

After running the audit cells, share:

1. `results/levir_audit/split_manifest.csv`
2. `results/levir_audit/image_mask_quality_sample.csv`
3. `results/levir_audit/training_examples.png`
4. Any warnings/error text (especially mismatched filenames or mask problems).

**Do not report pilot F1 or IoU as a final result.** If you decide to make this your term project, next step is a reproducible full U-Net baseline with locked splits and a Siamese U-Net comparison. Cite the dataset creators and respect the academic-only usage terms.